# NYT News 2000-2007 Source Corpus for Pragma-Stratified Fact Extraction

This notebook demonstrates a dataset of New York Times news articles (2000-2007) prepared as source documents for pragma-stratified fact extraction. The dataset contains 55,250 articles formatted with:

- **input**: Full article text (up to 3,500 characters)
- **output**: Article title
- **metadata**: Document type, source dataset, title, date, row index, text length

Articles range from 200 to 11,000+ characters and cover diverse topics (politics, culture, sports, business, lifestyle) from 2000-2007. This curated corpus supports extraction of pragmatic fact tiers:
- **Assertions**: Direct claims made in the text
- **Presuppositions**: Background assumptions underlying claims
- **Implicatures**: Conversational inferences not explicitly stated

Each article typically contains 8-10 extractable facts across these tiers, making it ideal for training and evaluating pragma-stratified fact extraction systems.

## Setup: Install Dependencies

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

## Imports

In [ ]:
import json
import sys
import hashlib
from pathlib import Path
from loguru import logger
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

## Data Loading

Load the mini demo dataset from GitHub (with local fallback for offline use).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-09ca95-pragma-stratified-problog-grounding-hall/main/round-2/dataset-1/demo/mini_demo_data.json"
import os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local filesystem")

## Load Data

In [ ]:
data = load_data()
logger.info(f"Loaded {len(data['datasets'][0]['examples'])} examples from mini_demo_data.json")

## Configuration

Define all tunable parameters here. Start with MINIMUM values for the demo:
- `MIN_CHARS`: Minimum article length to include
- `MAX_CHARS`: Maximum length to truncate at
- `PART_SIZE`: Number of examples per output file (demo: 1 part)

To run with original values, uncomment the commented lines below.

In [ ]:
# MINIMUM CONFIG FOR DEMO (fast, produces output)
MIN_CHARS = 200
MAX_CHARS = 3500
PART_SIZE = 3  # Process all 3 mini examples as one part

# Original production values (uncomment to use full dataset):
# MIN_CHARS = 200
# MAX_CHARS = 3500
# PART_SIZE = 27625

logger.info(f"Config: MIN_CHARS={MIN_CHARS}, MAX_CHARS={MAX_CHARS}, PART_SIZE={PART_SIZE}")

## Processing: Extract and Validate Examples

Filter examples based on text length constraints and prepare for output.

In [ ]:
# Extract examples from loaded data
examples = data['datasets'][0]['examples']
logger.info(f"Total examples in mini dataset: {len(examples)}")

# Filter by text length
filtered_examples = []
for i, example in enumerate(examples):
    text_len = example.get('metadata_text_length', 0)
    if MIN_CHARS <= text_len <= 3500:  # Note: metadata already shows original length
        filtered_examples.append(example)
        logger.debug(f"Example {i}: title='{example.get('output', 'N/A')[:50]}...', length={text_len}")

logger.info(f"Filtered examples: {len(filtered_examples)} match MIN_CHARS >= {MIN_CHARS}")

## Splitting: Prepare Output Parts

Split filtered examples into parts for tractable processing (following original data.py logic).

In [ ]:
# Build output structure matching original schema
output_metadata = data['metadata']
dataset_name = data['datasets'][0]['dataset']

# Split into parts (demo: all in one part)
parts = [filtered_examples[i:i+PART_SIZE] for i in range(0, len(filtered_examples), PART_SIZE)]
logger.info(f"Split {len(filtered_examples)} examples into {len(parts)} part(s)")

# Prepare part outputs
part_outputs = []
for j, part in enumerate(parts, 1):
    part_data = {
        **output_metadata,
        "datasets": [{
            "dataset": dataset_name,
            "examples": part
        }]
    }
    part_outputs.append(part_data)
    logger.info(f"Part {j}: {len(part)} examples")

logger.info(f"Prepared {len(part_outputs)} output part(s) for visualization")

## Validation: Check Schema Compliance

Verify that each example has required fields.

In [ ]:
# Validate schema
required_fields = ['input', 'output', 'metadata_document_type', 'metadata_source_dataset']
all_valid = True

for i, example in enumerate(filtered_examples):
    for field in required_fields:
        if field not in example:
            logger.warning(f"Example {i}: missing field '{field}'")
            all_valid = False

if all_valid:
    logger.info(f"✓ All {len(filtered_examples)} examples pass schema validation")
else:
    logger.warning(f"✗ Some examples failed schema validation")

## Results: Data Summary & Visualization

Display key statistics about the processed dataset.

In [ ]:
# Extract metadata for visualization
titles = [ex.get('output', 'N/A')[:50] for ex in filtered_examples]
text_lengths = [ex.get('metadata_text_length', 0) for ex in filtered_examples]
dates = [ex.get('metadata_date', 'N/A') for ex in filtered_examples]

# Create a summary table
summary_df = pd.DataFrame({
    'Example': range(1, len(filtered_examples) + 1),
    'Title': titles,
    'Year': dates,
    'Text Length': text_lengths,
    'Document Type': [ex.get('metadata_document_type', 'N/A') for ex in filtered_examples]
})

print("\n=== DATASET SUMMARY ===")
print(f"Total examples processed: {len(filtered_examples)}")
print(f"Average text length: {np.mean(text_lengths):.0f} characters")
print(f"Min/Max text length: {np.min(text_lengths)}/{np.max(text_lengths)} characters")
print(f"\nExample Articles:")
print(summary_df.to_string(index=False))

## Visualization: Text Length Distribution

In [ ]:
# Plot text length distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of text lengths
ax1.bar(range(len(text_lengths)), text_lengths, color='steelblue', alpha=0.7, edgecolor='black')
ax1.axhline(np.mean(text_lengths), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(text_lengths):.0f}')
ax1.set_xlabel('Example Index', fontsize=11)
ax1.set_ylabel('Text Length (characters)', fontsize=11)
ax1.set_title('Article Text Lengths', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Summary statistics box
stats_text = f"""Dataset Statistics:
Total Examples: {len(filtered_examples)}
Avg Length: {np.mean(text_lengths):.0f} chars
Median Length: {np.median(text_lengths):.0f} chars
Min Length: {np.min(text_lengths)} chars
Max Length: {np.max(text_lengths)} chars

Document Type: News
Source: NYT 2000-2007
Years Covered: {', '.join(sorted(set(dates)))}"""

ax2.text(0.1, 0.5, stats_text, fontsize=11, verticalalignment='center',
         family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax2.axis('off')

plt.tight_layout()
plt.show()

logger.info(f"✓ Visualization complete. Processed {len(filtered_examples)} examples.")